# ✅ Health Check — Validate Your Setup

This notebook tests every Azure connection to make sure everything works.
Run all cells — every one should show ✅.

---

In [ ]:
import os
from dotenv import load_dotenv

# Load environment
load_dotenv("../.env")

print("📋 Checking .env file...")
required_vars = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_KEY",
    "AZURE_SEARCH_ENDPOINT",
    "AZURE_COSMOS_ENDPOINT",
    "AZURE_CONTENT_SAFETY_ENDPOINT",
]

all_ok = True
for var in required_vars:
    val = os.getenv(var)
    if val:
        # Show just the domain for endpoints, masked for keys
        display = val[:40] + "..." if len(val) > 40 else val
        print(f"  ✅ {var}: {display}")
    else:
        print(f"  ❌ {var}: NOT SET")
        all_ok = False

if all_ok:
    print("\n✅ All environment variables loaded!")
else:
    print("\n❌ Some variables are missing. Run setup.ipynb first.")

## Test 1: Azure OpenAI (GPT-5.2)

In [ ]:
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
)

# Test GPT-5.2
try:
    response = client.chat.completions.create(
        model=os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT52", "gpt-52"),
        messages=[{"role": "user", "content": "Say 'hello' in one word."}],
        max_tokens=10
    )
    print(f"✅ GPT-5.2: {response.choices[0].message.content}")
except Exception as e:
    print(f"❌ GPT-5.2: {e}")

# Test GPT-4o-mini
try:
    response = client.chat.completions.create(
        model=os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT4O_MINI", "gpt-4o-mini"),
        messages=[{"role": "user", "content": "Say 'hello' in one word."}],
        max_tokens=10
    )
    print(f"✅ GPT-4o-mini: {response.choices[0].message.content}")
except Exception as e:
    print(f"❌ GPT-4o-mini: {e}")

## Test 2: Azure OpenAI Embeddings

In [ ]:
try:
    response = client.embeddings.create(
        model=os.getenv("AZURE_OPENAI_DEPLOYMENT_EMBEDDING", "text-embedding-3-large"),
        input="Hello world"
    )
    dim = len(response.data[0].embedding)
    print(f"✅ Embeddings: {dim} dimensions (text-embedding-3-large)")
except Exception as e:
    print(f"❌ Embeddings: {e}")

## Test 3: Azure AI Search

In [ ]:
import urllib.request
import json

search_endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
search_key = os.getenv("AZURE_SEARCH_API_KEY")

try:
    url = f"{search_endpoint}/indexes?api-version=2023-11-01"
    req = urllib.request.Request(url, headers={"api-key": search_key})
    with urllib.request.urlopen(req, timeout=10) as resp:
        data = json.loads(resp.read())
        print(f"✅ Azure AI Search: Connected ({len(data.get('value', []))} indexes)")
except Exception as e:
    print(f"❌ Azure AI Search: {e}")

## Test 4: Azure Cosmos DB

In [ ]:
try:
    from azure.cosmos import CosmosClient
    
    cosmos_client = CosmosClient(
        url=os.getenv("AZURE_COSMOS_ENDPOINT"),
        credential=os.getenv("AZURE_COSMOS_KEY")
    )
    
    db = cosmos_client.get_database_client(os.getenv("AZURE_COSMOS_DATABASE", "agent-platform"))
    containers = list(db.list_containers())
    container_names = [c['id'] for c in containers]
    print(f"✅ Cosmos DB: Connected (containers: {', '.join(container_names)})")
except ImportError:
    print("⚠️ Cosmos DB: azure-cosmos package not installed. Run: pip install azure-cosmos")
except Exception as e:
    print(f"❌ Cosmos DB: {e}")

## Test 5: Azure AI Content Safety

In [ ]:
try:
    safety_endpoint = os.getenv("AZURE_CONTENT_SAFETY_ENDPOINT")
    safety_key = os.getenv("AZURE_CONTENT_SAFETY_KEY")
    
    url = f"{safety_endpoint}/contentsafety/text:analyze?api-version=2024-09-01"
    data = json.dumps({"text": "Hello, this is a test."}).encode()
    req = urllib.request.Request(
        url, data=data,
        headers={"Ocp-Apim-Subscription-Key": safety_key, "Content-Type": "application/json"}
    )
    with urllib.request.urlopen(req, timeout=10) as resp:
        result = json.loads(resp.read())
        print(f"✅ Content Safety: Connected (categories analyzed: {len(result.get('categoriesAnalysis', []))})")
except Exception as e:
    print(f"❌ Content Safety: {e}")

## Test 6: LangChain + LangGraph

In [ ]:
try:
    import langchain
    import langgraph
    import langchain_openai
    print(f"✅ LangChain: v{langchain.__version__}")
    print(f"✅ LangGraph: v{langgraph.__version__}")
    print(f"✅ LangChain-OpenAI: v{langchain_openai.__version__}")
except ImportError as e:
    print(f"❌ Missing package: {e}")
    print("   Run: pip install -r ../requirements.txt")

## Test 7: LangChain with Azure OpenAI

In [ ]:
from langchain_openai import AzureChatOpenAI

try:
    llm = AzureChatOpenAI(
        azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT52", "gpt-52"),
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
    )
    response = llm.invoke("Say 'LangChain works!' in exactly those words.")
    print(f"✅ LangChain + Azure OpenAI: {response.content}")
except Exception as e:
    print(f"❌ LangChain + Azure OpenAI: {e}")

---

## 📊 Summary

In [ ]:
print("\n" + "=" * 50)
print("📊 HEALTH CHECK SUMMARY")
print("=" * 50)
print()
print("If all tests show ✅, you're ready to start!")
print()
print("Next step:")
print("  📖 Open ../lab-01-react-agent/README.md (read first!)")
print("  🧪 Then open ../lab-01-react-agent/lab.ipynb")
print()
print("Happy learning! 🚀")